In [ ]:
from google.cloud import bigquery

PROJECT_ID  = "qwiklabs-gcp-00-c521a9ba0b6e"
DATASET_ID  = "fraud_detection"
RAW_TABLE   = "fraud_data_raw"
TRAIN_TABLE = "fraud_training_data"

client = bigquery.Client(project=PROJECT_ID)
print("Client ready.")

Client ready.


In [ ]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset `{DATASET_ID}` ready.")

# Loading the CSV from GCS into BigQuery
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(
    "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=job_config,
)
load_job.result()

table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}")
print(f"Loaded {table.num_rows:,} rows into `{RAW_TABLE}`.")

Dataset `fraud_detection` ready.
Loaded 50,000 rows into `fraud_data_raw`.


In [ ]:
#Previewing the data just to check
df = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 5
""").to_dataframe()

df

,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0


In [ ]:
sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}` AS

SELECT
    -- Original columns kept as-is
    Applicant_ID,
    Age,
    Income,
    Number_of_Dependents,
    Amount_Requested,
    Application_Frequency_Last_Year,
    IP_Address,
    Application_Date,
    Previous_Assistance_Date,

    -- (a) One-hot encode Employment_Status
    IF(Employment_Status = 'Employed', 1, 0) AS Employment_Status_Employed,
    IF(Employment_Status = 'Self-Employed', 1, 0) AS Employment_Status_Self_Employed,
    IF(Employment_Status = 'Unemployed', 1, 0) AS Employment_Status_Unemployed,

    -- (a) One-hot encode Device_Type
    IF(Device_Type = 'Desktop', 1, 0) AS Device_Type_Desktop,
    IF(Device_Type = 'Mobile', 1, 0) AS Device_Type_Mobile,
    IF(Device_Type = 'Tablet', 1, 0) AS Device_Type_Tablet,

    -- (b) Break Age into bins, then one-hot encode
    CASE
        WHEN Age BETWEEN 18 AND 24 THEN '18-24'
        WHEN Age BETWEEN 25 AND 34 THEN '25-34'
        WHEN Age BETWEEN 35 AND 44 THEN '35-44'
        WHEN Age BETWEEN 45 AND 54 THEN '45-54'
        WHEN Age BETWEEN 55 AND 64 THEN '55-64'
        ELSE '65+'
    END AS Age_Bin,

    IF(Age BETWEEN 18 AND 24, 1, 0) AS Age_Bin_18_24,
    IF(Age BETWEEN 25 AND 34, 1, 0) AS Age_Bin_25_34,
    IF(Age BETWEEN 35 AND 44, 1, 0) AS Age_Bin_35_44,
    IF(Age BETWEEN 45 AND 54, 1, 0) AS Age_Bin_45_54,
    IF(Age BETWEEN 55 AND 64, 1, 0) AS Age_Bin_55_64,
    IF(Age >= 65, 1, 0) AS Age_Bin_65_Plus,

    -- (c) Income-to-Amount-Requested ratio
    SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,

    -- (d) Days since previous assistance (NULL when no prior assistance)
    DATE_DIFF(
        CAST(Application_Date AS DATE),
        CAST(Previous_Assistance_Date AS DATE),
        DAY
    ) AS Time_Since_Previous_Assistance,

    -- (e) Convert True/False fields to 0s and 1s
    --     BigQuery autodetected these as BOOL, so we cast directly
    CAST(Previous_Assistance_Received AS INT64) AS Previous_Assistance_Received,
    CAST(Supporting_Doc_Verified AS INT64) AS Supporting_Doc_Verified,

    -- Fraudulent is already 0/1 in the source data
    CAST(Fraudulent AS INT64) AS Fraudulent

FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
"""

client.query(sql).result()

result = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}")
print(f"fraud_training_data created — {result.num_rows:,} rows, {len(result.schema)} columns.")

fraud_training_data created — 50,000 rows, 27 columns.


In [ ]:
#Verifying the output
df_out = client.query(f"""
    SELECT
        Age,
        Age_Bin,
        Age_Bin_25_34,
        Employment_Status_Employed,
        Employment_Status_Self_Employed,
        Employment_Status_Unemployed,
        Device_Type_Mobile,
        Income,
        Amount_Requested,
        Income_to_Amount_Requested,
        Previous_Assistance_Date,
        Time_Since_Previous_Assistance,
        Previous_Assistance_Received,
        Supporting_Doc_Verified,
        Fraudulent
    FROM `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}`
    LIMIT 10
""").to_dataframe()

df_out

,Age,Age_Bin,Age_Bin_25_34,Employment_Status_Employed,Employment_Status_Self_Employed,Employment_Status_Unemployed,Device_Type_Mobile,Income,Amount_Requested,Income_to_Amount_Requested,Previous_Assistance_Date,Time_Since_Previous_Assistance,Previous_Assistance_Received,Supporting_Doc_Verified,Fraudulent
0,19,18-24,0,0,1,0,1,22795,4240,5.376179,NaT,<NA>,0,0,0
1,23,18-24,0,0,1,0,0,66021,4478,14.743412,NaT,<NA>,0,1,0
2,19,18-24,0,0,0,1,1,42021,5867,7.162264,NaT,<NA>,0,1,0
3,21,18-24,0,0,0,1,0,0,3937,0.000000,2022-12-29,377,1,0,0
4,19,18-24,0,0,1,0,1,44333,6831,6.489972,2023-07-12,219,1,0,0
5,24,18-24,0,0,1,0,0,0,1618,0.000000,2023-07-26,185,1,1,0
6,20,18-24,0,0,0,1,1,37889,4771,7.941522,2023-07-31,349,1,1,0
7,23,18-24,0,0,0,1,1,0,9448,0.000000,2023-08-28,303,1,1,0
8,24,18-24,0,0,1,0,1,0,7037,0.000000,2024-10-15,32,1,1,0
9,23,18-24,0,0,1,0,1,0,4917,0.000000,NaT,<NA>,0,1,0


In [ ]:
df_out = client.query(f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATASET_ID}.{TRAIN_TABLE}`
    LIMIT 10
""").to_dataframe()

df_out

,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Application_Frequency_Last_Year,IP_Address,Application_Date,Previous_Assistance_Date,Employment_Status_Employed,...,Age_Bin_25_34,Age_Bin_35_44,Age_Bin_45_54,Age_Bin_55_64,Age_Bin_65_Plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance,Previous_Assistance_Received,Supporting_Doc_Verified,Fraudulent
0,7243,19,22795,3,4240,1,42.196.11.240,2024-11-02,NaT,0,...,0,0,0,0,0,5.376179,<NA>,0,0,0
1,9087,23,66021,4,4478,1,81.255.201.241,2024-12-14,NaT,0,...,0,0,0,0,0,14.743412,<NA>,0,1,0
2,40774,19,42021,1,5867,1,147.25.255.215,2024-12-28,NaT,0,...,0,0,0,0,0,7.162264,<NA>,0,1,0
3,32654,21,0,4,3937,1,26.131.99.29,2024-01-10,2022-12-29,0,...,0,0,0,0,0,0.000000,377,1,0,0
4,17981,19,44333,1,6831,1,22.187.33.231,2024-02-16,2023-07-12,0,...,0,0,0,0,0,6.489972,219,1,0,0
5,6416,24,0,4,1618,1,140.20.3.159,2024-01-27,2023-07-26,0,...,0,0,0,0,0,0.000000,185,1,1,0
6,23607,20,37889,5,4771,1,26.131.166.227,2024-07-14,2023-07-31,0,...,0,0,0,0,0,7.941522,349,1,1,0
7,25684,23,0,4,9448,1,184.225.252.174,2024-06-26,2023-08-28,0,...,0,0,0,0,0,0.000000,303,1,1,0
8,41197,24,0,0,7037,1,88.236.169.207,2024-11-16,2024-10-15,0,...,0,0,0,0,0,0.000000,32,1,1,0
9,10152,23,0,0,4917,2,22.187.81.113,2024-05-17,NaT,0,...,0,0,0,0,0,0.000000,<NA>,0,1,0
